In [1]:
import pyreadstat
import pandas as pd
import numpy as np
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# Path to your .sav file
file_path = "Caffeine_Demographics_including_ITT.sav"

# Read the file
df, meta = pyreadstat.read_sav(file_path)

# Keep only specific patients
patients_to_keep = {
    'Group A': ['CA-03', 'CA-05', 'CA-07', 'CA-10', 'CA-11', 'CA-13', 'CA-15',
                'CA-18', 'CA-19', 'CA-26', 'CA-27', 'CA-38', 'CA-40', 'CA-42',
                'CA-52', 'CA-55', 'CA-63', 'CA-64', 'CA-66', 'CA-71'],
    'Group B': ['CA-01', 'CA-02', 'CA-04', 'CA-06', 'CA-09', 'CA-14', 'CA-21',
                'CA-24', 'CA-25', 'CA-28', 'CA-29', 'CA-34', 'CA-36', 'CA-48',
                'CA-49', 'CA-53', 'CA-54', 'CA-56', 'CA-65', 'CA-69', 'CA-70']
}

# Optional: override Group column if you want to redefine group assignments
df = df[df['StudyID'].isin(sum(patients_to_keep.values(), []))].copy()

# Save as CSV for easier re-upload
df.to_csv("caffeine_demographics.csv", index=False)

df.head(100)

,StudyID,Age,Group,Gender,WeightKg,ASAScore,AsianRace,CaucasianRace,Hispanic,NonHispanic,...,LapTransverseColectomy,LapSigmoidColectomy,LapPartialColectomy,LapProctocolecotomy,LapLAR,LapColostomy,LapAPR,LapSBR,LapIleostomy,LapAppy
0,CA-01,26.0,2.0,0.0,107.5,2.0,0.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CA-02,19.0,2.0,0.0,97.5,2.0,0.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CA-03,73.0,1.0,1.0,66.0,2.0,0.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
3,CA-04,59.0,2.0,1.0,80.0,3.0,0.0,1.0,0.0,1.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CA-05,57.0,1.0,0.0,69.0,2.0,0.0,1.0,0.0,1.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CA-06,79.0,2.0,1.0,56.7,3.0,0.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,CA-07,34.0,1.0,0.0,94.0,3.0,0.0,1.0,0.0,1.0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,CA-09,57.0,2.0,0.0,112.0,2.0,0.0,1.0,0.0,1.0,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CA-10,54.0,1.0,0.0,91.0,2.0,0.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,CA-11,46.0,1.0,1.0,100.0,2.0,0.0,1.0,0.0,1.0,...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Table 1 generator matching the example layout
# Save as make_table1.py and run in the same folder as caffeine_demographics.csv

CSV_PATH = "caffeine_demographics.csv"
OUT_CSV  = "table1_demographics_full.csv"
OUT_XLSX = "table1_demographics_full.xlsx"

# -------------------------- helpers --------------------------

def find_first_col(df, candidates):
    """Return first column from `candidates` that exists in df (case-insensitive)."""
    # normalize: map lowercase stripped names to original
    norm_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in norm_map:
            return norm_map[key]
    return None

def is_binary_series(s):
    vals = pd.unique(s.dropna())
    return set(vals).issubset({0, 1, True, False})

def fmt_count_pct(count, n):
    pct = 100.0 * count / n if n else 0.0
    return f"{int(count)} ({pct:.0f})"

def fmt_mean_sd(series):
    return f"{series.mean():.0f} ± {series.std(ddof=1):.0f}"

def fmt_mean_range(series):
    s = series.dropna().astype(float)
    if s.empty:
        return "—"
    return f"{s.mean():.0f} [{s.min():.0f}–{s.max():.0f}]"

def fmt_median_iqr(series):
    s = series.dropna().astype(float)
    if s.empty:
        return "—"
    q1, q2, q3 = np.percentile(s, [25, 50, 75])
    return f"{q2:.0f} [{q1:.0f}–{q3:.0f}]"

def asd_cont(x1, x2):
    """Absolute standardized difference (%) for continuous vars based on mean/SD."""
    m1, m2 = x1.mean(), x2.mean()
    v1, v2 = x1.var(ddof=1), x2.var(ddof=1)
    denom = np.sqrt((v1 + v2) / 2.0)
    if denom == 0 or np.isnan(denom):
        return 0.0
    return abs((m1 - m2) / denom) * 100.0

def asd_bin(p1, p2):
    """Absolute standardized difference (%) for binary vars using proportions."""
    denom = np.sqrt((p1*(1-p1) + p2*(1-p2)) / 2.0)
    if denom == 0 or np.isnan(denom):
        return 0.0
    return abs((p1 - p2) / denom) * 100.0

def header_row(title):
    return [title, "", "", ""]

def overall_asd_multilevel(level_props_g1, level_props_g2):
    """Overall ASD for multi-level categorical: max ASD across levels (common practice in Table 1s)."""
    asds = []
    for p1, p2 in zip(level_props_g1, level_props_g2):
        asds.append(asd_bin(p1, p2))
    return max(asds) if asds else 0.0

# ---------------------- load & light prep ---------------------

df = pd.read_csv(CSV_PATH)

# Map group & gender if coded numerically
GROUP_MAP  = {1: "Placebo", 2: "Caffeine"}
GENDER_MAP = {0: "Male", 1: "Female"}

if df["Group"].dtype != object:
    df["Group"] = df["Group"].map(GROUP_MAP)
if "Gender" in df.columns and df["Gender"].dtype != object:
    df["Gender"] = df["Gender"].map(GENDER_MAP)

g1 = df[df["Group"] == "Placebo"].copy()
g2 = df[df["Group"] == "Caffeine"].copy()
N1, N2 = len(g1), len(g2)

# ---------------------- variable sets ------------------------

# Demographics (continuous mean±SD)
DEM_CONT = [
    ("Age",                "Age, mean (SD), y"),
    ("WeightKg",           "Weight, mean (SD), kg"),
]

# Male sex
MALE_COL = "Gender"   # expects 'Male'/'Female'

# Race/Ethnicity binary flags (0/1)
RACE_ETH = [
    (["CaucasianRace", "White", "Race_White"], "White"),
    (["AsianRace", "Asian", "Race_Asian"], "Asian"),
    (["Hispanic", "HispanicEthnicity"], "Hispanic ethnicity"),
    (["NonHispanic", "NotHispanic"], "Non-Hispanic"),
]

# Education (binary if available)
EDU = (["CollegeOrHigher", "CollegeEducation", "CollegeEdu", "EducationCollegePlus"], "College education or higher")

# ASA (1/2/3)
ASA_COL = find_first_col(df, ["ASAScore", "ASA", "ASA_Score"])
ASA_LEVELS = [1, 2, 3]

# Comorbidities (binary flags; add/remove names to match your sheet)
COMORBIDITY_CANDIDATES = [
    (["Anxiety"], "Anxiety"),
    (["AtrialFibrillation", "AFib"], "Atrial fibrillation"),
    (["CoronaryArteryDisease", "CAD"], "Coronary artery disease"),
    (["CerebrovascularDisease", "CVDx", "CVA_TIA"], "Cerebrovascular disease"),
    (["CongestiveHeartFailure", "CHF"], "Congestive heart failure"),
    (["ChronicKidneyDisease", "CKD"], "Chronic kidney disease"),
    (["COPD"], "COPD"),
    (["CrohnDisease", "Crohns"], "Crohn disease"),
    (["Depression"], "Depression"),
    (["Hypertension", "HTN"], "Hypertension"),
    (["Malignancy", "Cancer"], "Malignancy"),
    (["ObstructiveSleepApnea", "OSA"], "Obstructive sleep apnea"),
    (["UlcerativeColitis", "UC"], "Ulcerative colitis"),
]

# Surgical procedures (binary flags); also create subtotals like in the figure
# We will compute:
#   - Colorectal subtotal = any of colectomy/proctocolectomy/rectal/LAR/APR/colostomy/ileostomy
#   - Small bowel subtotal = any of small bowel resection (SBR) / 'SmallBowel'
SURGICAL_ALL = [c for c in df.columns if c.lower().startswith("lap")]
# classify patterns (case-insensitive)
def any_of(df, cols):
    if not cols:
        return pd.Series(False, index=df.index)
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return pd.Series(False, index=df.index)
    m = df[cols].fillna(0).astype(int)
    return (m.sum(axis=1) > 0)

colorectal_cols = [c for c in SURGICAL_ALL if any(k in c.lower()
                      for k in ["colect", "procto", "rect", "lar", "apr", "colostomy", "ileostomy"])]
small_bowel_cols = [c for c in SURGICAL_ALL if any(k in c.lower()
                      for k in ["sbr", "smallbowel", "small_bowel"])]

# Baseline scores (continuous; display median[IQR], compute ASD on raw values)
PAIN_REST   = find_first_col(df, ["Pain_resting", "Pain_Resting", "PainResting", "VASRest"])
PAIN_DEEP   = find_first_col(df, ["Pain_deep_breath", "Pain_DeepBreath", "PainDeepBreath", "VASBreath"])
PAIN_MOVE   = find_first_col(df, ["Pain_movement", "Pain_Movement", "PainMovement", "VASMovement"])
PANAS_POS   = find_first_col(df, ["PANAS_Positive", "PANAS_Pos", "PANAS_Pos_score", "PANASP"])
PANAS_NEG   = find_first_col(df, ["PANAS_Negative", "PANAS_Neg", "PANAS_Neg_score", "PANASN"])
RCSQ        = find_first_col(df, ["RCSQ", "RCSQ_Total", "RC_Sleep_Quality"])
CAFF_INTAKE = find_first_col(df, ["Caffeine_Intake", "CaffeineIntake", "Caffeine_Intake_Baseline"])

# ---------------------- build table rows ---------------------

rows = []

# Demographics
# rows.append(header_row("Demographics"))
# for col, label in DEM_CONT:
#     if col not in df.columns:
#         continue
#     rows.append([
#         label,
#         fmt_mean_sd(g1[col]),
#         fmt_mean_sd(g2[col]),
#         f"{asd_cont(g1[col], g2[col]):.0f}",
#     ])
# Demographics
rows.append(header_row("Demographics"))
for col, label in DEM_CONT:
    if col not in df.columns:
        continue

    if col == "Age":
        placebo_val = fmt_median_iqr(g1[col])
        caffeine_val = fmt_median_iqr(g2[col])
        label = "Age, median [IQR], y"
    else:
        placebo_val = fmt_mean_sd(g1[col])
        caffeine_val = fmt_mean_sd(g2[col])

    rows.append([
        label,
        placebo_val,
        caffeine_val,
        f"{asd_cont(g1[col], g2[col]):.0f}",
    ])


# Male sex
if MALE_COL in df.columns:
    male1 = (g1[MALE_COL] == "Male").sum()
    male2 = (g2[MALE_COL] == "Male").sum()
    p1, p2 = (male1 / N1 if N1 else 0.0), (male2 / N2 if N2 else 0.0)
    rows.append([
        "Male sex",
        fmt_count_pct(male1, N1),
        fmt_count_pct(male2, N2),
        f"{asd_bin(p1, p2):.0f}",
    ])

# Race / Ethnicity
present_race = []
for cands, label in RACE_ETH:
    col = find_first_col(df, cands)
    if col is not None and is_binary_series(df[col]):
        present_race.append((col, label))

if present_race:
    rows.append(header_row("Race / Ethnicity"))
    # Level rows with individual ASDs
    level_p1, level_p2 = [], []
    for col, label in present_race:
        n1 = g1[col].fillna(0).astype(int).sum()
        n2 = g2[col].fillna(0).astype(int).sum()
        p1 = n1 / N1 if N1 else 0.0
        p2 = n2 / N2 if N2 else 0.0
        level_p1.append(p1); level_p2.append(p2)
        rows.append([f"  {label}", fmt_count_pct(n1, N1), fmt_count_pct(n2, N2), f"{asd_bin(p1, p2):.0f}"])
    # Optional overall line (uncomment to match your preferred style)
    # rows.append(["Race / Ethnicity (overall)", "", "", f"{overall_asd_multilevel(level_p1, level_p2):.0f}"])

# Education
edu_col = find_first_col(df, EDU[0])
if edu_col and is_binary_series(df[edu_col]):
    n1 = g1[edu_col].fillna(0).astype(int).sum()
    n2 = g2[edu_col].fillna(0).astype(int).sum()
    p1, p2 = (n1 / N1 if N1 else 0.0), (n2 / N2 if N2 else 0.0)
    rows.append([
        EDU[1],
        fmt_count_pct(n1, N1),
        fmt_count_pct(n2, N2),
        f"{asd_bin(p1, p2):.0f}",
    ])

# ASA physical status (with subrows 1/2/3; ASD shown per level)
if ASA_COL:
    rows.append(header_row("ASA physical status"))
    for lvl in ASA_LEVELS:
        n1 = (g1[ASA_COL] == lvl).sum()
        n2 = (g2[ASA_COL] == lvl).sum()
        p1, p2 = (n1 / N1 if N1 else 0.0), (n2 / N2 if N2 else 0.0)
        rows.append([f"  {lvl}", fmt_count_pct(n1, N1), fmt_count_pct(n2, N2), f"{asd_bin(p1, p2):.0f}"])

# Comorbidities
present_comorb = []
for cands, label in COMORBIDITY_CANDIDATES:
    col = find_first_col(df, cands)
    if col and is_binary_series(df[col]):
        present_comorb.append((col, label))

if present_comorb:
    rows.append(header_row("Comorbidities"))
    for col, label in present_comorb:
        n1 = g1[col].fillna(0).astype(int).sum()
        n2 = g2[col].fillna(0).astype(int).sum()
        p1, p2 = (n1 / N1 if N1 else 0.0), (n2 / N2 if N2 else 0.0)
        rows.append([f"  {label}", fmt_count_pct(n1, N1), fmt_count_pct(n2, N2), f"{asd_bin(p1, p2):.0f}"])

# Surgical procedure subtotals (now with ASD calculation)
if colorectal_cols or small_bowel_cols:
    rows.append(header_row("Surgical procedure"))
    if colorectal_cols:
        any_colorectal_g1 = any_of(g1, colorectal_cols)
        any_colorectal_g2 = any_of(g2, colorectal_cols)
        n1 = any_colorectal_g1.sum()
        n2 = any_colorectal_g2.sum()
        p1 = n1 / N1 if N1 else 0.0
        p2 = n2 / N2 if N2 else 0.0
        asd_val = asd_bin(p1, p2)
        rows.append([
            "  Colorectal",
            fmt_count_pct(n1, N1),
            fmt_count_pct(n2, N2),
            f"{asd_val:.0f}"
        ])
    if small_bowel_cols:
        any_small_g1 = any_of(g1, small_bowel_cols)
        any_small_g2 = any_of(g2, small_bowel_cols)
        n1 = any_small_g1.sum()
        n2 = any_small_g2.sum()
        p1 = n1 / N1 if N1 else 0.0
        p2 = n2 / N2 if N2 else 0.0
        asd_val = asd_bin(p1, p2)
        rows.append([
            "  Small bowel",
            fmt_count_pct(n1, N1),
            fmt_count_pct(n2, N2),
            f"{asd_val:.0f}"
        ])

# Baseline scores (median [IQR] display)
baseline_rows = []
if PAIN_REST: baseline_rows.append((PAIN_REST, "Pain—resting, median [IQR], mm"))
if PAIN_DEEP: baseline_rows.append((PAIN_DEEP, "Pain—deep breath, median [IQR], mm"))
if PAIN_MOVE: baseline_rows.append((PAIN_MOVE, "Pain—movement, median [IQR], mm"))
if PANAS_POS: baseline_rows.append((PANAS_POS, "PANAS—positive, median [IQR], score"))
if PANAS_NEG: baseline_rows.append((PANAS_NEG, "PANAS—negative, median [IQR], score"))
if RCSQ:      baseline_rows.append((RCSQ,      "RCSQ, mean (SD), score"))
if CAFF_INTAKE: baseline_rows.append((CAFF_INTAKE, "Caffeine intake—median [IQR]"))

if baseline_rows:
    rows.append(header_row("Baseline scores"))
    for col, label in baseline_rows:
        # decide format: RCSQ as mean±SD (per screenshot), others as median[IQR]
        if "RCSQ" in label:
            p_str = fmt_mean_sd(g1[col])
            c_str = fmt_mean_sd(g2[col])
        else:
            p_str = fmt_median_iqr(g1[col])
            c_str = fmt_median_iqr(g2[col])
        # ASD computed with mean/SD even if displayed as median[IQR]
        asd = asd_cont(g1[col].astype(float), g2[col].astype(float))
        rows.append([label, p_str, c_str, f"{asd:.0f}"])

# ---------------------- assemble & save ----------------------

table = pd.DataFrame(
    rows,
    columns=[
        "Characteristic",
        f"Placebo (N = {N1})",
        f"Caffeine (N = {N2})",
        "Absolute standardized difference (%)",
    ],
)

table.to_csv(OUT_CSV, index=False)
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    table.to_excel(w, index=False, sheet_name="Table1")

print(table.to_string(index=False))
print(f"\nSaved: {OUT_CSV} and {OUT_XLSX}")

                     Characteristic Placebo (N = 20) Caffeine (N = 21) Absolute standardized difference (%)
                       Demographics                                                                        
               Age, median [IQR], y       48 [40–57]        59 [42–66]                                   30
              Weight, mean (SD), kg          80 ± 15           84 ± 20                                   20
                           Male sex          10 (50)           11 (52)                                    5
                   Race / Ethnicity                                                                        
                              White         20 (100)           20 (95)                                   32
                              Asian            0 (0)             1 (5)                                   32
                 Hispanic ethnicity            1 (5)             0 (0)                                   32
                       Non-H

In [ ]:
# make_word_table.py
# Creates a Word table styled like your screenshot from the CSV produced earlier.
CSV_IN  = "table1_demographics_full.csv"
DOC_OUT = "Table1_Participant_Characteristics_2.docx"

# ---------- helpers ----------
def set_cell_shading(cell, fill_hex: str):
    """Set cell background color (hex without #)."""
    tcPr = cell._tc.get_or_add_tcPr()
    shd = tcPr.find(qn('w:shd'))
    if shd is None:
        shd = OxmlElement('w:shd')
        tcPr.append(shd)
    shd.set(qn('w:fill'), fill_hex)

def set_cell_borders(cell, color="000000"):
    """Thin borders around a cell."""
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = tcPr.find(qn('w:tcBorders'))
    if tcBorders is None:
        tcBorders = OxmlElement('w:tcBorders')
        tcPr.append(tcBorders)
    for tag in ("top", "left", "bottom", "right"):
        el = tcBorders.find(qn(f"w:{tag}"))
        if el is None:
            el = OxmlElement(f"w:{tag}")
            tcBorders.append(el)
        el.set(qn("w:val"), "single")
        el.set(qn("w:sz"), "6")         # 0.5pt
        el.set(qn("w:color"), color)

def add_para(cell, text, bold=False, align=None, indent_in=0.0):
    p = cell.paragraphs[0] if cell.paragraphs else cell.add_paragraph()
    p.clear() if p.runs else None
    run = p.add_run(text)
    run.bold = bold
    if align:
        p.alignment = align
    # left indent in inches
    if indent_in:
        p.paragraph_format.left_indent = Inches(indent_in)
    return p

def looks_like_section(row):
    # In the CSV we wrote, section header rows have blank numeric columns
    return (not str(row[1]).strip()) and (not str(row[2]).strip())

# ---------- load table data ----------
df = pd.read_csv(CSV_IN)

# Expect columns:
# "Characteristic", "Placebo (N = ...)", "Caffeine (N = ...)", "Absolute standardized difference (%)"
col_char = df.columns[0]
col_pbo  = df.columns[1]
col_caf  = df.columns[2]
col_asd  = df.columns[3]

# ---------- build document ----------
doc = Document()

# Title band (like "Table 1. Participant Characteristics")
title_tbl = doc.add_table(rows=1, cols=1)
title_tbl.alignment = WD_TABLE_ALIGNMENT.LEFT
cell = title_tbl.rows[0].cells[0]
set_cell_shading(cell, "77621F")  # warm gold/brown
set_cell_borders(cell, color="77621F")
p = add_para(cell, "Table 1. Participant Characteristics", bold=True)
p.runs[0].font.color.rgb = None
p.runs[0].font.size = Pt(12)
p.alignment = WD_ALIGN_PARAGRAPH.LEFT

doc.add_paragraph()  # spacer

# Main table
tbl = doc.add_table(rows=1, cols=4)
tbl.alignment = WD_TABLE_ALIGNMENT.LEFT
tbl.autofit = False

# Set column widths (tweak if needed)
tbl.columns[0].width = Inches(3.6)  # Characteristic
tbl.columns[1].width = Inches(1.6)  # Placebo
tbl.columns[2].width = Inches(1.6)  # Caffeine
tbl.columns[3].width = Inches(2.0)  # ASD

# Header row
hdr = tbl.rows[0].cells
hdr[0].text = col_char
hdr[1].text = col_pbo
hdr[2].text = col_caf
hdr[3].text = "Absolute standardized difference (%)"

for c in hdr:
    set_cell_shading(c, "E6E1D1")  # light beige
    set_cell_borders(c)
    for r in c.paragraphs:
        r.runs[0].font.bold = True

# Body rows
for _, r in df.iterrows():
    row = tbl.add_row().cells
    char = str(r[col_char])
    pbo  = "" if pd.isna(r[col_pbo]) else str(r[col_pbo])
    caf  = "" if pd.isna(r[col_caf]) else str(r[col_caf])
    asd  = "" if pd.isna(r[col_asd]) else str(r[col_asd])

    # Section headers get bold + subtle shading
    if looks_like_section([char, pbo, caf, asd]):
        add_para(row[0], char, bold=True)
        set_cell_shading(row[0], "F6F4ED")
        for j in range(1,4):
            add_para(row[j], "", bold=False)
            set_cell_shading(row[j], "F6F4ED")
    else:
        # Indent subrows that start with two spaces (we wrote "  label")
        indent = 0.25 if char.startswith("  ") else 0.0
        add_para(row[0], char, bold=False, indent_in=indent)

        # Right‑align numeric columns
        add_para(row[1], pbo, align=WD_ALIGN_PARAGRAPH.RIGHT)
        add_para(row[2], caf, align=WD_ALIGN_PARAGRAPH.RIGHT)
        add_para(row[3], asd, align=WD_ALIGN_PARAGRAPH.RIGHT)

    # borders + vertical alignment
    for c in row:
        set_cell_borders(c)
        c.vertical_alignment = WD_ALIGN_VERTICAL.CENTER

# Optional footnote (edit to your study’s wording)
doc.add_paragraph()
foot = doc.add_paragraph(
    "Data are presented as number (%) unless otherwise specified. "
    "ASA, American Society of Anesthesiologists; RCSQ, Richards–Campbell Sleep Questionnaire; "
    "PANAS, Positive and Negative Affect Schedule; SD, standard deviation."
)
foot.runs[0].font.size = Pt(9)

doc.save(DOC_OUT)
print(f"Saved: {DOC_OUT}")


Saved: Table1_Participant_Characteristics.docx
